# Make EFT scaling matrix

In [ ]:
from coffea.util import load

In [ ]:
f = load("TTHSMEFTtest/output_all.coffea")

In [ ]:
# this is how to access the column
f['columns']['ttHSMEFT__genMatch']['TTH-SMEFT_2024']['btag_mask']['nominal']['events_EFTfitCoefficients']

In [ ]:
import numpy as np
import json
import awkward as ak
# import coffea.util # Uncomment if loading a .coffea file directly

# Load your coffea output dictionary here
# f = coffea.util.load("my_output.coffea") 

# For exploration, let's define the base path to your specific process and systematic
process_name = 'TTH-SMEFT_2024'
base_path = f['columns']['ttHSMEFT__genMatch'][process_name]['btag_mask']['nominal']

# Let's see what columns are actually available in this path
print("Available columns in this path:")
print(list(base_path.keys()))

In [ ]:
# 1. Extract the raw numpy arrays from the column_accumulators
coef_array = base_path['events_EFTfitCoefficients'].value
idx1_array = base_path['events_EFTfitCoefficientIndex1'].value[0] 
idx2_array = base_path['events_EFTfitCoefficientIndex2'].value[0] 

# Grab the packed integer array for the names (just the first row)
raw_wc_names = base_path['events_WCnames'].value[0]

# 2. Extract your kinematic variables for masking (adjust names based on your keys)
# pt_array = base_path['events_Zh_pt'].value
# n_jets = base_path['events_n_ak4jets'].value

print(f"Coefficient array shape: {coef_array.shape}")
print(f"Index 1 array length: {len(idx1_array)}")
print(f"Raw WC names array length: {len(raw_wc_names)}")

# Quick sanity check: Do the number of coefficients match the number of indices?
assert coef_array.shape[1] == len(idx1_array) == len(idx2_array), "Mismatch in coefficient/index lengths!"
print("Array shapes look good!")

In [ ]:
def decode_WCnames(WCnames):
    WCnames_out = []
    WCname = ''
    for val in WCnames:
        fourchar = int(val).to_bytes(4, 'big').decode()
        if '-' in fourchar:
            WCname += fourchar.split('-')[-1]
        else:
            if len(WCname):
                WCnames_out.append(WCname)
            WCname = fourchar.lstrip('\x00')
    WCnames_out.append(WCname)
    return WCnames_out

def extract_eft_parameters(coef_array, idx1_array, idx2_array, process_name, bin_dict, wc_names, threshold=1e-4):
    out_dict = {process_name: {}}
    
    for bin_name, mask in bin_dict.items():
        print(f"Processing {bin_name}...")
        
        masked_weights = coef_array[mask]
        sum_weights = np.sum(masked_weights, axis=0)
        
        sm_mask = (idx1_array == 0) & (idx2_array == 0)
        sm_yield = sum_weights[sm_mask][0]
        
        if sm_yield == 0:
            print(f"  Warning: SM yield is 0 in {bin_name}. Skipping.")
            continue
            
        norm_weights = sum_weights / sm_yield
        sig_mask = np.abs(norm_weights) >= threshold
        
        bin_results = {
            "sm_yield": float(sm_yield),
            "linear": {},
            "quadratic": {}
        }
        
        for k in np.where(sig_mask)[0]:
            i, j = idx1_array[k], idx2_array[k]
            val = float(norm_weights[k])
            
            if i == 0 and j == 0:
                continue 
            elif j == 0:
                bin_results["linear"][wc_names[i]] = val
            else:
                wc_name1, wc_name2 = wc_names[i], wc_names[j]
                term_name = f"{wc_name1}_{wc_name2}" if wc_name1 <= wc_name2 else f"{wc_name2}_{wc_name1}"
                bin_results["quadratic"][term_name] = val
                
        out_dict[process_name][bin_name] = bin_results
        print(f"  -> SM Yield: {sm_yield:.2f} | Kept {len(bin_results['linear'])} linear, {len(bin_results['quadratic'])} quadratic terms.")
        
    return out_dict

In [ ]:
def format_for_combine(eft_data_dict, process_name, channel_name, target_bins, parameters):
    """
    Converts our dictionary output into the rigid Combine JSON format.
    
    Args:
        eft_data_dict: The dictionary output from our previous extract function.
        process_name: String (e.g., 'ttH').
        channel_name: String representing the datacard bin/category (e.g., 'SR_2lss').
        target_bins: List of string names mapping to the histogram bins in order 
                     (e.g., ['pt_200_300', 'pt_300_450']).
        parameters: Strict list of WC names you are passing to Combine.
    """
    process_data = eft_data_dict.get(process_name, {})
    
    combine_scaling_matrix = []
    
    # Combine loops over histogram bins in the channel FIRST
    for bin_name in target_bins:
        bin_results = process_data.get(bin_name)
        if not bin_results:
            print(f"Warning: Bin {bin_name} not found in data. Filling with 0s.")
            bin_results = {"linear": {}, "quadratic": {}}
            
        binscaling = []
        ncoef = len(parameters) + 1 # +1 for the implicit SM at index 0
        
        # Combine's exact nested loop
        for icoef in range(ncoef):
            for jcoef in range(icoef + 1):
                
                # 1. SM term
                if icoef == 0 and jcoef == 0:
                    val = 1.0 # Combine expects relative scaling, SM is 1.0
                    
                # 2. Linear term
                elif jcoef == 0:
                    wc = parameters[icoef - 1]
                    val = bin_results["linear"].get(wc, 0.0)
                    
                # 3. Quadratic / Cross term
                else:
                    wc1 = parameters[icoef - 1]
                    wc2 = parameters[jcoef - 1]
                    # Respect alphabetical ordering to match our dictionary keys
                    term_name = f"{wc1}_{wc2}" if wc1 <= wc2 else f"{wc2}_{wc1}"
                    val = bin_results["quadratic"].get(term_name, 0.0)
                    
                binscaling.append(val)
                
        combine_scaling_matrix.append(binscaling)

    # Construct the final JSON object for this process/channel
    combine_json_entry = {
        "channel": channel_name,
        "process": process_name,
        "parameters": parameters,
        "scaling": combine_scaling_matrix
    }
    
    return combine_json_entry

In [ ]:
# 1. Decode your Wilson Coefficient names and add "SM" at index 0
wc_names = ["SM"] + decode_WCnames(raw_wc_names)
print(f"Successfully decoded {len(wc_names)-1} Wilson Coefficients.")
print(f"First 5 WC names: {wc_names[:5]}")

# 2. Build boolean masks for your phase space bins
inclusive_mask = np.ones(coef_array.shape[0], dtype=bool)
my_bins = {"inclusive": inclusive_mask}

# 3. Run the extractor
eft_data = extract_eft_parameters(coef_array, idx1_array, idx2_array, "ttH", my_bins, wc_names, threshold=0)

# 4. Take a quick look at the output
#print(json.dumps(eft_data, indent=2))

In [ ]:
# 1. Define your Wilson Coefficient names (Index 0 MUST be 'SM')
#wc_names = ["SM", "ctW", "ctZ", "ctp", "cpQM", ...] # Replace with your actual decoder list

# 2. Build boolean masks for your phase space bins
# Example using dummy kinematic arrays from Cell 2:
# baseline_cut = (n_jets >= 5)
# my_bins = {
#     "pt_200_300": baseline_cut & (pt_array >= 200) & (pt_array < 300),
#     "pt_300_450": baseline_cut & (pt_array >= 300) & (pt_array < 450),
#     "pt_450_inf": baseline_cut & (pt_array >= 450)
# }

# For testing right now, let's just make an inclusive bin using a mask of all True
inclusive_mask = np.ones(coef_array.shape[0], dtype=bool)
my_bins = {"inclusive": inclusive_mask}

# 3. Run the extractor
# eft_data = extract_eft_parameters(coef_array, idx1_array, idx2_array, "ttH", my_bins, wc_names, threshold=1e-4)

# 4. Take a quick look at the output structure
# print(json.dumps(eft_data, indent=2))

In [ ]:
# 1. Grab all decoded Wilson Coefficients, skipping the "SM" at index 0
my_combine_params = wc_names[1:]

print(f"Passing {len(my_combine_params)} parameters to Combine...")

# 2. Loop over your bins and generate the JSON entries
combine_json_output = []

for bin_name in my_bins.keys():
    entry = format_for_combine(
        eft_data_dict = eft_data, 
        process_name = "ttH", 
        channel_name = bin_name,  # Treat bin as channel
        target_bins = [bin_name], # 1 bin per channel
        parameters = my_combine_params
    )
    combine_json_output.append(entry)

# 3. Print or save
import json
#print(json.dumps(combine_json_output, indent=2))

#with open('combine_scaling_ttH.json', 'w') as f_out:
#     json.dump(combine_json_output, f_out, indent=4)

In [ ]:
def validate_scaling_matrix(combine_json_entry, bin_index=0):
    """
    Takes a single entry from the generated Combine JSON list and prints it 
    as a human-readable 2D lower triangular matrix.
    
    Args:
        combine_json_entry: A single dictionary from your combine_json_output list.
        bin_index: Which bin's scaling array to print (default is 0, the first bin).
    """
    channel = combine_json_entry["channel"]
    process = combine_json_entry["process"]
    parameters = combine_json_entry["parameters"]
    
    # Combine lists the parameters without SM, so we add it back for the labels
    labels = ["SM"] + parameters
    ncoef = len(labels)
    
    # Grab the flat scaling list for the requested bin
    flat_scaling = combine_json_entry["scaling"][bin_index]
    
    # 1. Reconstruct the 2D lower triangle from the flat list
    matrix = []
    idx = 0
    for i in range(ncoef):
        row = []
        for j in range(i + 1):
            row.append(flat_scaling[idx])
            idx += 1
        matrix.append(row)
        
    # 2. Formatting and Printing
    print(f"=== Scaling Matrix for Process: {process} | Channel: {channel} | Bin: {bin_index} ===")
    
    # Dynamic column width based on the longest parameter name
    col_width = max(max(len(lbl) for lbl in labels), 10)
    
    # Print Column Headers
    header = " " * col_width + " | " + " | ".join(f"{lbl:>{col_width}}" for lbl in labels)
    print(header)
    print("-" * len(header))
    
    # Print each row
    for i, row in enumerate(matrix):
        row_label = labels[i]
        
        # Format the numbers: Scientific notation for small EFT values, plain '0' for exact zeros
        row_strs = []
        for val in row:
            if val == 0.0 or val == 1.0:
                row_strs.append(f"{val:>{col_width}.1f}")
            else:
                row_strs.append(f"{val:>{col_width}.4e}")
                
        # Pad the missing upper triangle with blank spaces to emphasize the shape
        padding = [" " * col_width] * (ncoef - len(row))
        
        row_str = " | ".join(row_strs + padding)
        print(f"{row_label:>{col_width}} | {row_str}")
    print("\n")

In [ ]:
import numpy as np
import coffea.util
import json

# 1. Load the file and navigate to the process
f = coffea.util.load("TTBBEFT/output_all.coffea")
#process_name = 'TTH-SMEFT_2024'
#base_path = f['columns']['ttHSMEFT__genMatch'][process_name]['btag_mask']['nominal']

process_name = 'TTBB-SMEFT_2024'
base_path = f['columns']['ttbbSMEFT'][process_name]['btag_mask']['nominal']

# 2. Extract EFT weight arrays
coef_array = base_path['events_EFTfitCoefficients'].value
idx1_array = base_path['events_EFTfitCoefficientIndex1'].value[0]
idx2_array = base_path['events_EFTfitCoefficientIndex2'].value[0]
raw_wc_names = base_path['events_WCnames'].value[0]

# 3. Extract Kinematic and NN arrays 
# (Assuming the 'events_' prefix matches your coffea output structure)
n_ak4jets    = base_path['events_n_ak4jets'].value
n_b_outZH    = base_path['events_n_b_outZH'].value
ZH_bbvLscore = base_path['events_ZH_bbvLscore'].value
MET_pt       = base_path['events_MET_pt'].value
ZH_M         = base_path['events_ZH_M'].value
ZH_pt        = base_path['events_ZH_pt'].value
nnscore      = base_path['spanet_output_signal'].value 
# Add this to your variable extractions in Cell 1
if "TTH" in process_name:
    genZHpt = base_path['events_genZHpt'].value

# If your DataCardShapes script requires truth matching for the base mask, extract that too:
# matchedGen = base_path['events_matchedGen_ZHbb_bb'].value 

print(f"Successfully loaded {coef_array.shape[0]} events.")

In [ ]:
# Decoder function
def decode_WCnames(WCnames):
    WCnames_out = []
    WCname = ''
    for val in WCnames:
        fourchar = int(val).to_bytes(4, 'big').decode()
        if '-' in fourchar:
            WCname += fourchar.split('-')[-1]
        else:
            if len(WCname):
                WCnames_out.append(WCname)
            WCname = fourchar.lstrip('\x00')
    WCnames_out.append(WCname)
    return WCnames_out

wc_names = ["SM"] + decode_WCnames(raw_wc_names)

# Replicating your cuts() function exactly
baseline_mask = (
    (n_ak4jets >= 5) &
    (n_b_outZH == 2) &
    (ZH_bbvLscore >= 0.9870) &
    (MET_pt >= 20) &
    (ZH_M >= 50) & 
    (ZH_M <= 200) 
    # & (matchedGen == True) # Uncomment if required for the signal template
)

# Define pt channels
ch_masks = {
    "Zhpt1": baseline_mask & (ZH_pt >= 200) & (ZH_pt < 300),
    "Zhpt2": baseline_mask & (ZH_pt >= 300) & (ZH_pt < 450),
    "Zhpt3": baseline_mask & (ZH_pt >= 450)
}

# Your pt_bins from the datacard script
pt_bins = [0, 200, 300, 450]

# Define the generator-level process masks
if "TTH" in process_name:
    gen_masks = {
        "ttH0": (genZHpt >= pt_bins[0]) & (genZHpt < pt_bins[1]),
        "ttH1": (genZHpt >= pt_bins[1]) & (genZHpt < pt_bins[2]),
        "ttH2": (genZHpt >= pt_bins[2]) & (genZHpt < pt_bins[3]),
        "ttH3": (genZHpt >= pt_bins[3])
    }
else:
    # For TTBB (or other backgrounds), do not bin in genZHpt.
    # We create a single mask of all Trues to accept every event into the "ttbb" process template.
    gen_masks = {
        "ttbb": np.ones(len(n_ak4jets), dtype=bool)
    }

# Define Mass bins
mass_bins = [50, 80, 105, 145, 200]
nn_bins_dict = {
    "Zhpt1": [0.0, 0.00331425, 0.57303775, 0.76005952, 0.87137488, 0.93786220, 1.0], # Dummy values, replace these
    "Zhpt2": [0.0, 0.04882239, 0.82911105, 0.89332885, 0.94393039, 0.97217209, 1.0], 
    "Zhpt3": [0.0, 0.00927468, 0.76219768, 0.86899641, 0.93080776, 0.96656023, 1.0]
}

In [ ]:
def get_binned_eft_weights(nn_array, mass_array, coef_array, nn_edges, mass_edges):
    """Sorts events into 2D bins and sums the EFT coefficients for each bin."""
    nn_idx = np.digitize(nn_array, nn_edges) - 1
    mass_idx = np.digitize(mass_array, mass_edges) - 1
    
    n_nn, n_mass = len(nn_edges) - 1, len(mass_edges) - 1
    binned_weights = np.zeros((n_nn, n_mass, coef_array.shape[1]))
    
    valid_mask = (nn_idx >= 0) & (nn_idx < n_nn) & (mass_idx >= 0) & (mass_idx < n_mass)
    np.add.at(binned_weights, (nn_idx[valid_mask], mass_idx[valid_mask]), coef_array[valid_mask])
    
    return binned_weights

def unroll_and_merge_bins(binned_weights, channel_name):
    """Applies the MakeDataCard merging and flattening logic."""
    a = np.copy(binned_weights)
    if channel_name == "Zhpt1": # Equivalant to pt_bin == 0 in your code
        a[:, -2, :] = a[:, -2, :] + a[:, -1, :]
        return a[:, :-1, :].reshape(-1, a.shape[-1])
    else:
        return a.reshape(-1, a.shape[-1])

def create_channel_json(unrolled_weights, idx1_array, idx2_array, wc_names, combine_params, channel, process):
    """Formats the dense lower-triangular list for Combine."""
    
    # Map combine params back to NanoAOD indices by stripping the RooFit syntax
    param_to_nano_idx = []
    for p in combine_params:
        clean_name = p.split('[')[0] # e.g., 'cg[0,-10,10]' becomes 'cg'
        
        if clean_name == "SM":
            param_to_nano_idx.append(0)
        else:
            param_to_nano_idx.append(wc_names.index(clean_name))

    # Fast lookup for (i, j) pairs
    pair_to_k = { **{(idx1_array[k], idx2_array[k]): k for k in range(len(idx1_array))},
                  **{(idx2_array[k], idx1_array[k]): k for k in range(len(idx1_array))} }

    scaling = []
    ncoef = len(combine_params) 

    for b in range(unrolled_weights.shape[0]):
        bin_weights = unrolled_weights[b]
        sm_yield = bin_weights[pair_to_k[(0, 0)]]

        binscaling = []
        if sm_yield == 0:
            for i in range(ncoef):
                for j in range(i + 1):
                    binscaling.append(1.0 if (i==0 and j==0) else 0.0)
            scaling.append(binscaling)
            continue

        norm_weights = bin_weights / sm_yield
        
        for i in range(ncoef):
            for j in range(i + 1):
                nano_i = param_to_nano_idx[i]
                nano_j = param_to_nano_idx[j]
                
                if nano_i == 0 and nano_j == 0:
                    binscaling.append(1.0) 
                else:
                    k = pair_to_k.get((nano_i, nano_j))
                    val = float(norm_weights[k]) if k is not None else 0.0
                    binscaling.append(val if abs(val) >= 1e-4 else 0.0)
                    
        scaling.append(binscaling)

    return {
        "channel": channel,
        "process": process,
        "parameters": combine_params,
        "scaling": scaling
    }

In [ ]:
combine_json_output = []

# 1. Define the specific WCs you want to include in the scaling matrix
wcs_to_keep = ["cthre", "chtbre", "chq3", 'ctgre', 'cbwre', 'chq1', 'cht', 'ctwre', 'ctbre'] # Replace with your actual WCs

# 2. Build the parameters list using RooFit factory syntax for ONLY those WCs
my_combine_params = ["SM[1]"] 
for wc in wcs_to_keep:
    if wc in wc_names: # Safety check to ensure the WC exists in the file
        my_combine_params.append(f"{wc}[0,-20,20]")
    else:
        print(f"Warning: {wc} not found in the raw WC names!")

# 1. Loop over reconstructed channels (the datacard bins)
for channel_name, ch_mask in ch_masks.items():
    print(f"\nProcessing Channel: {channel_name}...")
    
    # Renamed loop variable to 'gen_proc_name' to avoid overwriting global 'process_name'
    for gen_proc_name, gen_mask in gen_masks.items():
        
        final_mask = ch_mask & gen_mask
        
        n_passing = np.sum(final_mask)
        if n_passing == 0:
            print(f"  -> Skipping {gen_proc_name} (0 events fall into this channel).")
            continue
            
        print(f"  -> Extracting {gen_proc_name} ({n_passing} events)...")
        
        binned_weights = get_binned_eft_weights(
            nnscore[final_mask], ZH_M[final_mask], coef_array[final_mask], 
            nn_bins_dict[channel_name], mass_bins
        )
        
        unrolled_weights = unroll_and_merge_bins(binned_weights, channel_name)
        
        entry = create_channel_json(
            unrolled_weights, idx1_array, idx2_array, 
            wc_names, my_combine_params, 
            channel=channel_name, process=gen_proc_name
        )
        combine_json_output.append(entry)

import json
out_filename = f'combine_scaling_{"ttH" if "TTH" in process_name else "ttbb"}.json'

with open(out_filename, 'w') as f_out:
    json.dump(combine_json_output, f_out, indent=4)
    
print(f"\nFinished! File saved as '{out_filename}'")
print(combine_json_output)

In [ ]:
import numpy as np
import coffea.util
import json
import gc
import concurrent.futures
# ==========================================
# 1. Helper Functions
# ==========================================
def decode_WCnames(WCnames):
    WCnames_out = []
    WCname = ''
    for val in WCnames:
        fourchar = int(val).to_bytes(4, 'big').decode()
        if '-' in fourchar:
            WCname += fourchar.split('-')[-1]
        else:
            if len(WCname):
                WCnames_out.append(WCname)
            WCname = fourchar.lstrip('\x00')
    WCnames_out.append(WCname)
    return WCnames_out

def get_binned_eft_weights(nn_array, mass_array, coef_array, nn_edges, mass_edges):
    nn_idx = np.digitize(nn_array, nn_edges) - 1
    mass_idx = np.digitize(mass_array, mass_edges) - 1
    
    n_nn, n_mass = len(nn_edges) - 1, len(mass_edges) - 1
    binned_weights = np.zeros((n_nn, n_mass, coef_array.shape[1]))
    
    valid_mask = (nn_idx >= 0) & (nn_idx < n_nn) & (mass_idx >= 0) & (mass_idx < n_mass)
    np.add.at(binned_weights, (nn_idx[valid_mask], mass_idx[valid_mask]), coef_array[valid_mask])
    
    return binned_weights

def unroll_and_merge_bins(binned_weights, channel_name):
    a = np.copy(binned_weights)
    if channel_name == "Zhpt1": 
        a[:, -2, :] = a[:, -2, :] + a[:, -1, :]
        return a[:, :-1, :].reshape(-1, a.shape[-1])
    else:
        return a.reshape(-1, a.shape[-1])

def create_channel_json(unrolled_weights, idx1_array, idx2_array, wc_names, combine_params, channel, process):
    param_to_nano_idx = []
    for p in combine_params:
        clean_name = p.split('[')[0]
        if clean_name == "SM":
            param_to_nano_idx.append(0)
        else:
            param_to_nano_idx.append(wc_names.index(clean_name))

    pair_to_k = { **{(idx1_array[k], idx2_array[k]): k for k in range(len(idx1_array))},
                  **{(idx2_array[k], idx1_array[k]): k for k in range(len(idx1_array))} }

    scaling = []
    ncoef = len(combine_params) 

    for b in range(unrolled_weights.shape[0]):
        bin_weights = unrolled_weights[b]
        sm_yield = bin_weights[pair_to_k[(0, 0)]]

        binscaling = []
        if sm_yield == 0:
            for i in range(ncoef):
                for j in range(i + 1):
                    binscaling.append(1.0 if (i==0 and j==0) else 0.0)
            scaling.append(binscaling)
            continue

        norm_weights = bin_weights / sm_yield
        
        for i in range(ncoef):
            for j in range(i + 1):
                nano_i = param_to_nano_idx[i]
                nano_j = param_to_nano_idx[j]
                
                if nano_i == 0 and nano_j == 0:
                    binscaling.append(1.0) 
                else:
                    k = pair_to_k.get((nano_i, nano_j))
                    val = float(norm_weights[k]) if k is not None else 0.0
                    binscaling.append(val if abs(val) >= 1e-4 else 0.0)
                    
        scaling.append(binscaling)

    return {
        "channel": channel,
        "process": process,
        "parameters": combine_params,
        "scaling": scaling
    }

def process_single_file(proc_label, proc_info, my_combine_params_in, pt_bins, mass_bins, nn_bins_dict, wcs_to_keep):
    print(f"\n========================================")
    print(f"Loading Base Process: {proc_label} ({proc_info['name']})")
    print(f"From File: {proc_info['file']}")
    
    # 1. Load file
    f = coffea.util.load(proc_info['file'])
    base_path = f['columns'][proc_info['collection']][proc_info['name']]['btag_mask']['nominal']
    
    # 2. Extract arrays and DOWNCAST coefficients to save memory
    coef_array = base_path['events_EFTfitCoefficients'].value.astype(np.float32)
    idx1_array = base_path['events_EFTfitCoefficientIndex1'].value[0]
    idx2_array = base_path['events_EFTfitCoefficientIndex2'].value[0]
    raw_wc_names = base_path['events_WCnames'].value[0]
    
    n_ak4jets    = base_path['events_n_ak4jets'].value
    n_b_outZH    = base_path['events_n_b_outZH'].value
    ZH_bbvLscore = base_path['events_ZH_bbvLscore'].value
    MET_pt       = base_path['events_MET_pt'].value
    ZH_M         = base_path['events_ZH_M'].value
    ZH_pt        = base_path['events_ZH_pt'].value
    nnscore      = base_path['spanet_output_signal'].value 
    
    if proc_label == "ttH":
        genZHpt = base_path['events_genZHpt'].value 
    
    # 3. Destroy massive dictionary immediately
    del f
    del base_path
    gc.collect()

    wc_names = ["SM"] + decode_WCnames(raw_wc_names)
    
    # Setup params if this is the first process
    my_combine_params = my_combine_params_in
    if my_combine_params is None:
        my_combine_params = ["SM[1]"] 
        for wc in wcs_to_keep:
            if wc in wc_names:
                my_combine_params.append(f"{wc}[0,-20,20]")

    baseline_mask = (
        (n_ak4jets >= 5) & (n_b_outZH == 2) & (ZH_bbvLscore >= 0.9870) &
        (MET_pt >= 20) & (ZH_M >= 50) & (ZH_M <= 200) 
    )

    ch_masks = {
        "Zhpt1": baseline_mask & (ZH_pt >= 200) & (ZH_pt < 300),
        "Zhpt2": baseline_mask & (ZH_pt >= 300) & (ZH_pt < 450),
        "Zhpt3": baseline_mask & (ZH_pt >= 450)
    }

    if proc_label == "ttH":
        gen_masks = {
            "ttH0": (genZHpt >= pt_bins[0]) & (genZHpt < pt_bins[1]),
            "ttH1": (genZHpt >= pt_bins[1]) & (genZHpt < pt_bins[2]),
            "ttH2": (genZHpt >= pt_bins[2]) & (genZHpt < pt_bins[3]),
            "ttH3": (genZHpt >= pt_bins[3])
        }
    elif proc_label == "ttbb":
        gen_masks = {"ttbb": np.ones(len(n_ak4jets), dtype=bool)}
        
    local_json_output = []
    
    for channel_name, ch_mask in ch_masks.items():
        for gen_proc_name, gen_mask in gen_masks.items():
            final_mask = ch_mask & gen_mask
            n_passing = np.sum(final_mask)
            
            if n_passing == 0:
                continue
                
            binned_weights = get_binned_eft_weights(
                nnscore[final_mask], ZH_M[final_mask], coef_array[final_mask], 
                nn_bins_dict[channel_name], mass_bins
            )
            unrolled_weights = unroll_and_merge_bins(binned_weights, channel_name)
            
            entry = create_channel_json(
                unrolled_weights, idx1_array, idx2_array, 
                wc_names, my_combine_params, channel=channel_name, process=gen_proc_name
            )
            local_json_output.append(entry)
            
    # Return the generated JSON entries and the params list back to the main script
    return local_json_output, my_combine_params

# ==========================================
# 2. Main Script Setup
# ==========================================

# Dictionary mapping standard names to their specific coffea output paths and individual files
processes_to_run = {
    "ttH": {
        "file": "TTHSMEFTtest/output_all.coffea",  # <-- UPDATE with actual path for ttH
        "name": "TTH-SMEFT_2024",
        "collection": "ttHSMEFT__genMatch"
    },
    "ttbb": {
        "file": "TTBBEFT/output_all.coffea", # <-- UPDATE with actual path for ttbb
        "name": "TTBB-SMEFT_2024",
        "collection": "ttbbSMEFT"
    }
}

pt_bins = [0, 200, 300, 450]
mass_bins = [50, 80, 105, 145, 200]
nn_bins_dict = {
    "Zhpt1": [0.0, 0.00331425, 0.57303775, 0.76005952, 0.87137488, 0.93786220, 1.0], 
    "Zhpt2": [0.0, 0.04882239, 0.82911105, 0.89332885, 0.94393039, 0.97217209, 1.0], 
    "Zhpt3": [0.0, 0.00927468, 0.76219768, 0.86899641, 0.93080776, 0.96656023, 1.0]
}

wcs_to_keep = ["cthre", "chtbre", "chq3", 'ctgre', 'cbwre', 'chq1', 'cht', 'ctwre', 'ctbre'] 

# --- Execution ---
if __name__ == '__main__':
    combine_json_output = []
    my_combine_params = None 

    # Execute one file at a time in a separate process
    # max_workers=1 ensures we only unpickle one coffea file at a time
    with concurrent.futures.ProcessPoolExecutor(max_workers=1) as executor:
        for proc_label, proc_info in processes_to_run.items():
            
            # Submit the job to the isolated process
            future = executor.submit(
                process_single_file, 
                proc_label, 
                proc_info, 
                my_combine_params,
                pt_bins,
                mass_bins,
                nn_bins_dict,
                wcs_to_keep
            )
            
            # Wait for it to finish and get the results
            process_entries, updated_params = future.result()
            
            combine_json_output.extend(process_entries)
            
            # Update params so the next file uses the exact same Combine string list
            if my_combine_params is None:
                my_combine_params = updated_params

    # Save Final Multi-Process Matrix
    out_filename = 'combine_scaling_multiprocess.json'
    with open(out_filename, 'w') as f_out:
        json.dump(combine_json_output, f_out, indent=4)
        
    print(f"\nFinished! File saved as '{out_filename}'")

In [5]:
import numpy as np
import json
import gc
import concurrent.futures
import pyarrow.dataset as ds
import awkward as ak

# ==========================================
# 1. Helper Functions
# ==========================================
def decode_WCnames(WCnames):
    WCnames_out = []
    WCname = ''
    for val in WCnames:
        fourchar = int(val).to_bytes(4, 'big').decode()
        if '-' in fourchar:
            WCname += fourchar.split('-')[-1]
        else:
            if len(WCname):
                WCnames_out.append(WCname)
            WCname = fourchar.lstrip('\x00')
    WCnames_out.append(WCname)
    return WCnames_out

def get_binned_eft_weights(nn_array, mass_array, coef_array, nn_edges, mass_edges):
    nn_idx = np.digitize(nn_array, nn_edges) - 1
    mass_idx = np.digitize(mass_array, mass_edges) - 1
    
    n_nn, n_mass = len(nn_edges) - 1, len(mass_edges) - 1
    binned_weights = np.zeros((n_nn, n_mass, coef_array.shape[1]), dtype=np.float32)
    
    valid_mask = (nn_idx >= 0) & (nn_idx < n_nn) & (mass_idx >= 0) & (mass_idx < n_mass)
    np.add.at(binned_weights, (nn_idx[valid_mask], mass_idx[valid_mask]), coef_array[valid_mask])
    
    return binned_weights

def unroll_and_merge_bins(binned_weights, channel_name):
    a = np.copy(binned_weights)
    if channel_name == "Zhpt1": 
        a[:, -2, :] = a[:, -2, :] + a[:, -1, :]
        return a[:, :-1, :].reshape(-1, a.shape[-1])
    else:
        return a.reshape(-1, a.shape[-1])

def create_channel_json(unrolled_weights, idx1_array, idx2_array, wc_names, combine_params, channel, process):
    param_to_nano_idx = []
    for p in combine_params:
        clean_name = p.split('[')[0]
        if clean_name == "SM":
            param_to_nano_idx.append(0)
        else:
            param_to_nano_idx.append(wc_names.index(clean_name))

    pair_to_k = { **{(idx1_array[k], idx2_array[k]): k for k in range(len(idx1_array))},
                  **{(idx2_array[k], idx1_array[k]): k for k in range(len(idx1_array))} }

    scaling = []
    ncoef = len(combine_params) 

    for b in range(unrolled_weights.shape[0]):
        bin_weights = unrolled_weights[b]
        sm_yield = bin_weights[pair_to_k[(0, 0)]]

        binscaling = []
        if sm_yield == 0:
            for i in range(ncoef):
                for j in range(i + 1):
                    binscaling.append(1.0 if (i==0 and j==0) else 0.0)
            scaling.append(binscaling)
            continue

        norm_weights = bin_weights / sm_yield
        
        for i in range(ncoef):
            for j in range(i + 1):
                nano_i = param_to_nano_idx[i]
                nano_j = param_to_nano_idx[j]
                
                if nano_i == 0 and nano_j == 0:
                    binscaling.append(1.0) 
                else:
                    k = pair_to_k.get((nano_i, nano_j))
                    val = float(norm_weights[k]) if k is not None else 0.0
                    binscaling.append(val if abs(val) >= 1e-4 else 0.0)
                    
        scaling.append(binscaling)

    return {
        "channel": channel,
        "process": process,
        "parameters": combine_params,
        "scaling": scaling
    }


def process_parquet_dir(proc_label, proc_info, my_combine_params_in, pt_bins, mass_bins, nn_bins_dict, wcs_to_keep):
    print(f"\n========================================")
    print(f"Loading Parquet Directory: {proc_label} ({proc_info['name']})")
    print(f"Path: {proc_info['path']}")
    
    columns_to_read = [
        'events_EFTfitCoefficients', 'events_EFTfitCoefficientIndex1', 'events_EFTfitCoefficientIndex2', 'events_WCnames',
        'events_n_ak4jets', 'events_n_b_outZH', 'events_ZH_bbvLscore', 'events_MET_pt', 'events_ZH_M', 'events_ZH_pt',
        'spanet_output_signal'
    ]
    if proc_info['has_genZHpt']:
        columns_to_read.append('events_genZHpt')

    # 1. Load the dataset (Lazy Load)
    dataset = ds.dataset(proc_info['path'], format="parquet")
    
    # Accumulators to store weights across all batches
    accumulated_weights = {}
    
    idx1_array, idx2_array, wc_names = None, None, None
    my_combine_params = my_combine_params_in

    # 2. Process chunk by chunk
    for batch in dataset.to_batches(columns=columns_to_read):
        chunk = ak.from_arrow(batch)
        
        # Extract variables
        coef_array = ak.to_numpy(chunk['events_EFTfitCoefficients']).astype(np.float32)
        
        # We only need to pull the WC metadata from the very first event of the very first chunk
        if idx1_array is None:
            idx1_array = ak.to_numpy(chunk['events_EFTfitCoefficientIndex1'][0])
            idx2_array = ak.to_numpy(chunk['events_EFTfitCoefficientIndex2'][0])
            raw_wc_names = ak.to_list(chunk['events_WCnames'][0])
            wc_names = ["SM"] + decode_WCnames(raw_wc_names)
            
            if my_combine_params is None:
                my_combine_params = ["SM[1]"] 
                for wc in wcs_to_keep:
                    if wc in wc_names:
                        my_combine_params.append(f"{wc}[0,-20,20]")

        n_ak4jets    = ak.to_numpy(chunk['events_n_ak4jets'])
        n_b_outZH    = ak.to_numpy(chunk['events_n_b_outZH'])
        ZH_bbvLscore = ak.to_numpy(chunk['events_ZH_bbvLscore'])
        MET_pt       = ak.to_numpy(chunk['events_MET_pt'])
        ZH_M         = ak.to_numpy(chunk['events_ZH_M'])
        ZH_pt        = ak.to_numpy(chunk['events_ZH_pt'])
        nnscore      = ak.to_numpy(chunk['spanet_output_signal'])
        
        baseline_mask = (
            (n_ak4jets >= 5) & (n_b_outZH == 2) & (ZH_bbvLscore >= 0.9105) &
            (MET_pt >= 20) & (ZH_M >= 50) & (ZH_M <= 200) 
        )

        ch_masks = {
            "Zhpt1": baseline_mask & (ZH_pt >= 200) & (ZH_pt < 300),
            "Zhpt2": baseline_mask & (ZH_pt >= 300) & (ZH_pt < 450),
            "Zhpt3": baseline_mask & (ZH_pt >= 450)
        }

        if proc_info['has_genZHpt']:
            genZHpt = ak.to_numpy(chunk['events_genZHpt'])
            gen_masks = {
                "ttH0": (genZHpt >= pt_bins[0]) & (genZHpt < pt_bins[1]),
                "ttH1": (genZHpt >= pt_bins[1]) & (genZHpt < pt_bins[2]),
                "ttH2": (genZHpt >= pt_bins[2]) & (genZHpt < pt_bins[3]),
                "ttH3": (genZHpt >= pt_bins[3])
            }
        else:
            # Dynamically grab the output name so ttbb and ttH_non_genMatch are labeled correctly
            out_name = proc_info.get("output_name", proc_label)
            gen_masks = {out_name: np.ones(len(n_ak4jets), dtype=bool)}
            
        # Bin and Accumulate weights
        for channel_name, ch_mask in ch_masks.items():
            if channel_name not in accumulated_weights:
                accumulated_weights[channel_name] = {}
                
            for gen_proc_name, gen_mask in gen_masks.items():
                if gen_proc_name not in accumulated_weights[channel_name]:
                    # Initialize accumulator on first pass
                    n_nn = len(nn_bins_dict[channel_name]) - 1
                    n_mass = len(mass_bins) - 1
                    accumulated_weights[channel_name][gen_proc_name] = np.zeros((n_nn, n_mass, coef_array.shape[1]), dtype=np.float32)

                final_mask = ch_mask & gen_mask
                if np.sum(final_mask) == 0:
                    continue
                    
                binned_weights = get_binned_eft_weights(
                    nnscore[final_mask], ZH_M[final_mask], coef_array[final_mask], 
                    nn_bins_dict[channel_name], mass_bins
                )
                
                # Add to total process weight
                accumulated_weights[channel_name][gen_proc_name] += binned_weights

        # 3. Clean up batch memory explicitly
        del chunk, n_ak4jets, n_b_outZH, ZH_bbvLscore, MET_pt, ZH_M, ZH_pt, nnscore, coef_array, baseline_mask, ch_masks
        if proc_info['has_genZHpt']:
            del genZHpt
        gc.collect()

    # 4. Process accumulated weights into final JSON format
    local_json_output = []
    for channel_name, gen_dict in accumulated_weights.items():
        for gen_proc_name, binned_weights in gen_dict.items():
            # Skip empty bins to save space
            if np.sum(np.abs(binned_weights)) == 0:
                continue
                
            unrolled_weights = unroll_and_merge_bins(binned_weights, channel_name)
            
            entry = create_channel_json(
                unrolled_weights, idx1_array, idx2_array, 
                wc_names, my_combine_params, channel=channel_name, process=gen_proc_name
            )
            local_json_output.append(entry)
            
    return local_json_output, my_combine_params

# ==========================================
# 2. Main Script Setup
# ==========================================

# Updated to use Parquet chunk paths
processes_to_run = {
    "ttH_genMatch": {
        "path": "/cms/data/store/user/jsamudio/eftchunks/TTH-SMEFT_2024/genMatch/btag_mask/nominal",
        "name": "TTH-SMEFT_2024_gen",
        "has_genZHpt": True
    },
    "ttH_non_genMatch": {
        "path": "/cms/data/store/user/jsamudio/eftchunks/TTH-SMEFT_2024/non_genMatch/btag_mask/nominal",
        "name": "TTH-SMEFT_2024_nongen",
        "has_genZHpt": True,
        "output_name": "ttH_non_genMatch" # Labels this specific background in the JSON
    },
    "ttbb": {
        "path": "/cms/data/store/user/jsamudio/eftchunks/TTBB-SMEFT_2024/btag_mask/nominal",
        "name": "TTBB-SMEFT_2024",
        "has_genZHpt": False,
        "output_name": "tt_B" # Labels this specific background in the JSON
    }
}

pt_bins = [0, 200, 300, 450]
mass_bins = [50, 80, 105, 145, 200]
nn_bins_dict = {
    "Zhpt1": [0.0, 0.00331425, 0.57303775, 0.76005952, 0.87137488, 0.93786220, 1.0], 
    "Zhpt2": [0.0, 0.04882239, 0.82911105, 0.89332885, 0.94393039, 0.97217209, 1.0], 
    "Zhpt3": [0.0, 0.00927468, 0.76219768, 0.86899641, 0.93080776, 0.96656023, 1.0]
}

wcs_to_keep = ["cthre", "chtbre", "chq3", 'ctgre', 'cbwre', 'chq1', 'cht', 'ctwre', 'ctbre'] 

# --- Execution ---
if __name__ == '__main__':
    combine_json_output = []
    my_combine_params = None 

    # Because memory footprint per batch is so low, you can safely increase max_workers 
    # to your machine's CPU limits if you want to speed things up further.
    with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
        futures = []
        for proc_label, proc_info in processes_to_run.items():
            future = executor.submit(
                process_parquet_dir, 
                proc_label, 
                proc_info, 
                my_combine_params,
                pt_bins,
                mass_bins,
                nn_bins_dict,
                wcs_to_keep
            )
            futures.append(future)
            
        for future in concurrent.futures.as_completed(futures):
            process_entries, updated_params = future.result()
            combine_json_output.extend(process_entries)
            
            # Ensures subsequent files sync their parameter mapping list
            if my_combine_params is None:
                my_combine_params = updated_params

    # Save Final Matrix
    out_filename = 'combine_scaling_multiprocess.json'
    with open(out_filename, 'w') as f_out:
        json.dump(combine_json_output, f_out, indent=4)
        
    print(f"\nFinished! File saved as '{out_filename}'")



Loading Parquet Directory: ttH_genMatch (TTH-SMEFT_2024_gen)Loading Parquet Directory: ttH_non_genMatch (TTH-SMEFT_2024_nongen)

Path: /cms/data/store/user/jsamudio/eftchunks/TTH-SMEFT_2024/genMatch/btag_mask/nominalPath: /cms/data/store/user/jsamudio/eftchunks/TTH-SMEFT_2024/non_genMatch/btag_mask/nominal


Loading Parquet Directory: ttbb (TTBB-SMEFT_2024)
Path: /cms/data/store/user/jsamudio/eftchunks/TTBB-SMEFT_2024/btag_mask/nominal

Finished! File saved as 'combine_scaling_multiprocess.json'


In [4]:
print(combine_json_output)

[{'channel': 'Zhpt1', 'process': 'ttH1', 'parameters': ['SM[1]', 'cthre[0,-20,20]', 'chtbre[0,-20,20]', 'chq3[0,-20,20]', 'ctgre[0,-20,20]', 'cbwre[0,-20,20]', 'chq1[0,-20,20]', 'cht[0,-20,20]', 'ctwre[0,-20,20]', 'ctbre[0,-20,20]'], 'scaling': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,